In [66]:
from utils.DataPreprocessing import dataloader
from loguru import logger
import os

path = os.path.dirname(os.path.dirname(os.getcwd()))
print(path)
data = dataloader(path=path)

2024-03-05 00:37:23.052 | INFO     | utils.DataPreprocessing:load_csv:123 - shape: (11_103, 3)
┌───────────┬─────────────┬────────────┐
│ AVG_TEMP  ┆ AVG_TEMP_DC ┆ date       │
│ ---       ┆ ---         ┆ ---        │
│ f32       ┆ str         ┆ date       │
╞═══════════╪═════════════╪════════════╡
│ 29.299999 ┆ C           ┆ 1992-07-01 │
│ 29.200001 ┆ C           ┆ 1992-07-02 │
│ 29.6      ┆ C           ┆ 1992-07-03 │
│ 29.299999 ┆ C           ┆ 1992-07-04 │
│ 29.299999 ┆ C           ┆ 1992-07-05 │
│ …         ┆ …           ┆ …          │
│ 22.200001 ┆ C           ┆ 2022-11-26 │
│ 22.4      ┆ C           ┆ 2022-11-27 │
│ 25.4      ┆ C           ┆ 2022-11-28 │
│ 25.1      ┆ C           ┆ 2022-11-29 │
│ 22.1      ┆ C           ┆ 2022-11-30 │
└───────────┴─────────────┴────────────┘
2024-03-05 00:37:23.054 | INFO     | utils.DataPreprocessing:load_csv:125 - shape: (1, 3)
┌──────────┬─────────────┬──────┐
│ AVG_TEMP ┆ AVG_TEMP_DC ┆ date │
│ ---      ┆ ---         ┆ ---  │
│ u32      ┆ u32

/home/argonaut/programming/Global-Solar-Radiation-in-King-s-Park


In [67]:
logger.success(data.X_train)
logger.success(data.y_train)
logger.success(data.X_test)
logger.success(data.y_test)

2024-03-05 00:37:23.185 | SUCCESS  | __main__:<module>:1 - shape: (5_441, 34)
┌───────────┬─────┬──────┬─────┬───┬──────────────┬──────────────┬──────────────┬──────────────┐
│ GSR       ┆ SUN ┆ RH   ┆ UV  ┆ … ┆ AVG_TEMP_t-4 ┆ AVG_TEMP_t-3 ┆ AVG_TEMP_t-2 ┆ AVG_TEMP_t-1 │
│ ---       ┆ --- ┆ ---  ┆ --- ┆   ┆ ---          ┆ ---          ┆ ---          ┆ ---          │
│ f32       ┆ f32 ┆ f32  ┆ f32 ┆   ┆ f32          ┆ f32          ┆ f32          ┆ f32          │
╞═══════════╪═════╪══════╪═════╪═══╪══════════════╪══════════════╪══════════════╪══════════════╡
│ 16.440001 ┆ 6.9 ┆ 75.0 ┆ 4.0 ┆ … ┆ 28.799999    ┆ 29.0         ┆ 29.299999    ┆ 30.299999    │
│ 1.93      ┆ 0.0 ┆ 79.0 ┆ 0.6 ┆ … ┆ 29.0         ┆ 29.299999    ┆ 30.299999    ┆ 29.299999    │
│ 3.04      ┆ 0.0 ┆ 92.0 ┆ 1.0 ┆ … ┆ 29.299999    ┆ 30.299999    ┆ 29.299999    ┆ 25.9         │
│ 4.34      ┆ 0.0 ┆ 91.0 ┆ 1.0 ┆ … ┆ 30.299999    ┆ 29.299999    ┆ 25.9         ┆ 24.1         │
│ 2.04      ┆ 0.0 ┆ 93.0 ┆ 0.5 ┆ … ┆ 29.299999   

In [68]:
import polars as pl
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import lightgbm as lgb
 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
 
import warnings
warnings.filterwarnings('ignore')

In [69]:
data.y_train = data.y_train.to_numpy()
data.y_val = data.y_val.to_numpy()
data.y_test = data.y_test.to_numpy()

In [70]:
lgb_train = lgb.Dataset(data.X_train, data.y_train)
lgb_eval = lgb.Dataset(data.X_val, data.y_val, reference=lgb_train)

In [71]:
# Define a dictionary of parameters for configuring the LightGBM regression model.
# params = {
#     'objective': 'regression',
#     'metric': 'rmse',
#     'boosting_type': 'gbdt',
#     'num_leaves': 31,
#     'learning_rate': 0.05,
#     'feature_fraction': 0.9,
# }

# params = {
#     "boosting_type": "gbdt",
#     "objective": "regression",
#     "metric": {"l2", "l1"},
#     "num_leaves": 31,
#     "learning_rate": 0.05,
#     "feature_fraction": 0.9,
#     "bagging_fraction": 0.8,
#     "bagging_freq": 5,
#     "verbose": 1,
# }

params = {
    'task': 'train',
    "boosting_type": "gbdt",
    "objective": "regression",
    # "metric": {"l2", "l1"},
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": 1,
    "num_iterations": 10000
}

In [72]:
gbm = lgb.train(params,
    lgb_train,
    # num_boost_round=200,
    valid_sets=[lgb_train, lgb_eval],
    valid_names=['train','valid'],
    callbacks= [lgb.early_stopping(10)]
   )

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024838 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5640
[LightGBM] [Info] Number of data points in the train set: 5441, number of used features: 34
[LightGBM] [Info] Start training from score 23.068829


Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[136]	train's l2: 0.492163	valid's l2: 1.03683


In [73]:
##% Initial Models
from sklearn.ensemble import RandomForestRegressor
from sklearn import svm
import lightgbm as lgb
import xgboost as xg 

train_X = data.X_train
train_y = data.y_train

RFReg = RandomForestRegressor(random_state = 0).fit(train_X, train_y)
SVM = svm.SVR().fit(train_X, train_y) 
XGReg = xg.XGBRegressor(objective ='reg:squarederror', seed = 0,verbosity=0).fit(train_X,train_y) 
LGBMReg = lgb.LGBMRegressor(random_state=0).fit(train_X,train_y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000414 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5640
[LightGBM] [Info] Number of data points in the train set: 5441, number of used features: 34
[LightGBM] [Info] Start training from score 23.068829


In [74]:
##% evaluateRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluateRegressor(true, predicted, message = "Test set"):
    MSE = mean_squared_error(true, predicted,squared = True)
    MAE = mean_absolute_error(true, predicted)
    RMSE = mean_squared_error(true, predicted,squared = False)
    LogRMSE = mean_squared_error(np.log(true),np.log(predicted),squared = False)
    r2 = r2_score(true, predicted)
    print(message)
    print("MSE:", MSE)
    print("MAE:", MAE)
    print("RMSE:", RMSE)
    print("LogRMSE:", LogRMSE)
    print("R2:", r2)

In [75]:
##% Plot True vs predicted values. Useful for continuous y 
def PlotPrediction(true,predicted, title = "Dataset: "):
    fig = plt.figure(figsize=(20,20))
    ax1 = fig.add_subplot(111)
    ax1.set_title(title + 'True vs Predicted')
    ax1.scatter(list(range(0,len(true))),true, s=10, c='r', marker="o", label='True')
    ax1.scatter(list(range(0,len(predicted))), predicted, s=10, c='b', marker="o", label='Predicted')
    plt.legend(loc='upper right')
    plt.show()

In [76]:
valid_X = data.X_val
valid_y = data.y_val

##% Model Metrics
print("Random Forest Regressor") 
predicted_train_y = RFReg.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = RFReg.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")
print("\n")
    
print("Support Vector Machine") 
predicted_train_y = SVM.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = SVM.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")
print("\n")

print("XGBoost Regressor") 
predicted_train_y = XGReg.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = XGReg.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")
print("\n")

print("LightGBM Regressor") 
predicted_train_y = LGBMReg.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = LGBMReg.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")

Random Forest Regressor
    Training Set
MSE: 0.17111888620781734
MAE: 0.28810279563121377
RMSE: 0.4136651861201488
LogRMSE: 0.024361424773989927
R2: 0.9935640193942302
    Test Set
MSE: 1.2054097767399874
MAE: 0.7742071935133756
RMSE: 1.0979115523301444
LogRMSE: 0.06610027043575327
R2: 0.9582683821148879


Support Vector Machine
    Training Set
MSE: 17.233811146616063
MAE: 3.328612731867585
RMSE: 4.151362565064158
LogRMSE: 0.21180291637822055
R2: 0.35181629122797853
    Test Set
MSE: 17.832799414026077
MAE: 3.460244362495084
RMSE: 4.2228899362907955
LogRMSE: 0.21368658609237218
R2: 0.38262358135119523


XGBoost Regressor
    Training Set
MSE: 0.09191622
MAE: 0.22825725
RMSE: 0.30317688
LogRMSE: 0.014329148
R2: 0.9965429238839457
    Test Set
MSE: 1.2016491
MAE: 0.78318113
RMSE: 1.0961976
LogRMSE: 0.06514467
R2: 0.9583985797179244


LightGBM Regressor
    Training Set
MSE: 0.3666866316984105
MAE: 0.45690425191347406
RMSE: 0.6055465561774838
LogRMSE: 0.03106571231525574
R2: 0.986208488

In [79]:
valid_X = data.X_test
valid_y = data.y_test

##% Model Metrics
print("Random Forest Regressor") 
predicted_train_y = RFReg.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = RFReg.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")
print("\n")
    
print("Support Vector Machine") 
predicted_train_y = SVM.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = SVM.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")
print("\n")


print("XGBoost Regressor") 
predicted_train_y = XGReg.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = XGReg.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")
print("\n")

print("LightGBM Regressor") 
predicted_train_y = LGBMReg.predict(train_X)
evaluateRegressor(train_y,predicted_train_y,"    Training Set")
predicted_valid_y = LGBMReg.predict(valid_X)
evaluateRegressor(valid_y,predicted_valid_y,"    Test Set")

Random Forest Regressor
    Training Set
MSE: 0.17111888620781734
MAE: 0.28810279563121377
RMSE: 0.4136651861201488
LogRMSE: 0.024361424773989927
R2: 0.9935640193942302
    Test Set
MSE: 0.970533677751422
MAE: 0.7041152108998106
RMSE: 0.9851566767532066
LogRMSE: 0.05148189354662094
R2: 0.9559844516863404


Support Vector Machine
    Training Set
MSE: 17.233811146616063
MAE: 3.328612731867585
RMSE: 4.151362565064158
LogRMSE: 0.21180291637822055
R2: 0.35181629122797853
    Test Set
MSE: 14.317563069459263
MAE: 3.1350220478357405
RMSE: 3.78385558253209
LogRMSE: 0.17379954260837988
R2: 0.3506712817243822


XGBoost Regressor
    Training Set
MSE: 0.09191622
MAE: 0.22825725
RMSE: 0.30317688
LogRMSE: 0.014329148
R2: 0.9965429238839457
    Test Set
MSE: 1.0478593
MAE: 0.7547129
RMSE: 1.02365
LogRMSE: 0.053071298
R2: 0.9524775854067248


LightGBM Regressor
    Training Set
MSE: 0.3666866316984105
MAE: 0.45690425191347406
RMSE: 0.6055465561774838
LogRMSE: 0.03106571231525574
R2: 0.98620848871620

In [77]:
from sklearn.metrics import mean_squared_error

print("Starting predicting...")
# predict
y_pred = gbm.predict(data.X_test)
# eval
rmse_test = mean_squared_error(data.y_test, y_pred) ** 0.5
print(f"The RMSE of prediction is: {rmse_test}")

Starting predicting...
The RMSE of prediction is: 0.9360058580273364


In [78]:
# get best